# OMem Benchmark — Reproducible Results

This notebook reproduces the head-to-head benchmark numbers from the OMem README.

**What this measures:**
- Setup time (cold start)
- `add()` throughput (ops/second)
- RAG throughput (queries/second)
- RAG p99 latency (milliseconds)

**Systems compared:** OMem · ChromaDB · Mem0

**Dataset:** 5 000 memories, 500 queries, `all-MiniLM-L6-v2`, same random seed across all systems.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohitkumarrajbadi/omem/blob/main/benchmarks/reproduce.ipynb)

---

**Estimated runtime:** ~10–30 minutes depending on hardware.  
**Hardware note:** Original numbers were recorded on Apple M-series. Results will vary on other hardware but relative speedups should be consistent.

In [ ]:
# Cell 1 — Install dependencies
# Uncomment lines for systems you want to benchmark

!pip install omem-os --quiet
# !pip install chromadb --quiet
# !pip install mem0ai --quiet
# !pip install lancedb --quiet
!pip install pandas numpy --quiet

print('Installation complete.')

In [ ]:
# Cell 2 — Generate shared dataset
# Same seed used for all systems to ensure fair comparison.

import random
import time

SEED = 42
N_MEMORIES = 5_000
N_QUERIES = 500

random.seed(SEED)

TOPICS = [
    'database', 'API design', 'Python', 'TypeScript', 'deployment',
    'security', 'performance', 'testing', 'architecture', 'debugging',
    'user preferences', 'project decisions', 'bug reports', 'code review',
    'meetings', 'product requirements', 'infrastructure', 'monitoring',
]

TEMPLATES = [
    'We decided to use {topic} for the new feature because of scalability.',
    'Bug found in {topic}: null pointer exception in the main handler.',
    'User prefers {topic} over alternatives after testing.',
    'Architecture note: {topic} was chosen after evaluating three options.',
    'Action item from meeting: review {topic} implementation by Friday.',
    'Performance issue in {topic}: p99 latency exceeds 200ms under load.',
    'Security review for {topic}: passed with one minor recommendation.',
    'Documentation update needed for {topic} integration guide.',
    'Refactored {topic} module to reduce cyclomatic complexity.',
    'Test coverage for {topic} increased from 60% to 92%.',
]

memories = [
    random.choice(TEMPLATES).format(topic=random.choice(TOPICS))
    for _ in range(N_MEMORIES)
]

queries = [
    f'What decisions were made about {random.choice(TOPICS)}?'
    for _ in range(N_QUERIES)
]

print(f'Dataset ready: {len(memories)} memories, {len(queries)} queries')
print(f'Sample memory: {memories[0]}')
print(f'Sample query:  {queries[0]}')

In [ ]:
# Cell 3 — Benchmark OMem

import time
import numpy as np
from omem import OMem

omem_results = {}

# Setup time
t0 = time.perf_counter()
brain = OMem(backend='memory')
omem_results['setup_ms'] = (time.perf_counter() - t0) * 1000
print(f'Setup: {omem_results["setup_ms"]:.1f} ms')

# Add throughput
t0 = time.perf_counter()
for text in memories:
    brain.add(text)
elapsed = time.perf_counter() - t0
omem_results['add_ops_per_s'] = N_MEMORIES / elapsed
print(f'Add: {omem_results["add_ops_per_s"]:.1f} ops/s  ({elapsed:.1f}s total)')

# RAG throughput + p99
latencies = []
t0 = time.perf_counter()
for q in queries:
    qt = time.perf_counter()
    brain.recall(q, k=5)
    latencies.append((time.perf_counter() - qt) * 1000)
elapsed = time.perf_counter() - t0

omem_results['rag_ops_per_s'] = N_QUERIES / elapsed
omem_results['rag_p99_ms'] = float(np.percentile(latencies, 99))
omem_results['rag_p50_ms'] = float(np.percentile(latencies, 50))
print(f'RAG: {omem_results["rag_ops_per_s"]:.1f} ops/s  p50={omem_results["rag_p50_ms"]:.2f}ms  p99={omem_results["rag_p99_ms"]:.1f}ms')

print('\nOMem benchmark complete.')

In [ ]:
# Cell 4 — Benchmark ChromaDB and Mem0
# Uncomment the system(s) you want to compare against.

import numpy as np
import time

competitor_results = {}

# --- ChromaDB ---
try:
    import chromadb
    from chromadb.utils.embedding_functions import DefaultEmbeddingFunction

    t0 = time.perf_counter()
    client = chromadb.Client()
    col = client.create_collection('bench', embedding_function=DefaultEmbeddingFunction())
    competitor_results['chromadb_setup_ms'] = (time.perf_counter() - t0) * 1000

    t0 = time.perf_counter()
    batch = 500
    for i in range(0, N_MEMORIES, batch):
        chunk = memories[i:i+batch]
        col.add(documents=chunk, ids=[str(j) for j in range(i, i+len(chunk))])
    competitor_results['chromadb_add_ops_per_s'] = N_MEMORIES / (time.perf_counter() - t0)

    lats = []
    t0 = time.perf_counter()
    for q in queries:
        qt = time.perf_counter()
        col.query(query_texts=[q], n_results=5)
        lats.append((time.perf_counter() - qt) * 1000)
    competitor_results['chromadb_rag_ops_per_s'] = N_QUERIES / (time.perf_counter() - t0)
    competitor_results['chromadb_rag_p99_ms'] = float(np.percentile(lats, 99))
    print('ChromaDB benchmark complete.')

except ImportError:
    print('ChromaDB not installed. Skipping. Run: pip install chromadb')

# --- Mem0 ---
try:
    from mem0 import Memory as Mem0Memory

    t0 = time.perf_counter()
    m0 = Mem0Memory()
    competitor_results['mem0_setup_ms'] = (time.perf_counter() - t0) * 1000

    # Mem0 add is very slow — only sample 50 memories to avoid 30-minute wait
    sample = memories[:50]
    t0 = time.perf_counter()
    for text in sample:
        m0.add(text, user_id='bench')
    competitor_results['mem0_add_ops_per_s'] = len(sample) / (time.perf_counter() - t0)

    # RAG sample
    lats = []
    for q in queries[:50]:
        qt = time.perf_counter()
        m0.search(q, user_id='bench')
        lats.append((time.perf_counter() - qt) * 1000)
    competitor_results['mem0_rag_ops_per_s'] = 50 / sum(l/1000 for l in lats)
    competitor_results['mem0_rag_p99_ms'] = float(np.percentile(lats, 99))
    print('Mem0 benchmark complete (50-item sample).')

except ImportError:
    print('Mem0 not installed. Skipping. Run: pip install mem0ai')

print('Competitor benchmarks done.')

In [ ]:
# Cell 5 — Comparison Table

import pandas as pd

rows = [
    {
        'System': 'OMem',
        'Setup (ms)': round(omem_results.get('setup_ms', float('nan')), 1),
        'Add (ops/s)': round(omem_results.get('add_ops_per_s', float('nan')), 1),
        'RAG (ops/s)': round(omem_results.get('rag_ops_per_s', float('nan')), 1),
        'RAG p99 (ms)': round(omem_results.get('rag_p99_ms', float('nan')), 1),
        'RAG p50 (ms)': round(omem_results.get('rag_p50_ms', float('nan')), 3),
    },
]

if 'chromadb_setup_ms' in competitor_results:
    rows.append({
        'System': 'ChromaDB',
        'Setup (ms)': round(competitor_results.get('chromadb_setup_ms', float('nan')), 1),
        'Add (ops/s)': round(competitor_results.get('chromadb_add_ops_per_s', float('nan')), 1),
        'RAG (ops/s)': round(competitor_results.get('chromadb_rag_ops_per_s', float('nan')), 1),
        'RAG p99 (ms)': round(competitor_results.get('chromadb_rag_p99_ms', float('nan')), 1),
        'RAG p50 (ms)': float('nan'),
    })

if 'mem0_setup_ms' in competitor_results:
    rows.append({
        'System': 'Mem0',
        'Setup (ms)': round(competitor_results.get('mem0_setup_ms', float('nan')), 1),
        'Add (ops/s)': round(competitor_results.get('mem0_add_ops_per_s', float('nan')), 2),
        'RAG (ops/s)': round(competitor_results.get('mem0_rag_ops_per_s', float('nan')), 1),
        'RAG p99 (ms)': round(competitor_results.get('mem0_rag_p99_ms', float('nan')), 1),
        'RAG p50 (ms)': float('nan'),
    })

df = pd.DataFrame(rows)

print('=' * 70)
print('BENCHMARK RESULTS')
print('=' * 70)
print(df.to_string(index=False))
print()

# Speedup ratios vs OMem
omem_rag = omem_results.get('rag_ops_per_s', None)
if omem_rag:
    print('Speedup ratios (RAG throughput vs OMem):')
    for row in rows[1:]:
        system = row['System']
        rag = row['RAG (ops/s)']
        if rag and rag > 0:
            ratio = omem_rag / rag
            print(f'  OMem vs {system}: {ratio:.1f}x faster')

print()
print('Note: OMem add() includes embed+classify+dedup+graph sync.')
print('Competitor add() is raw vector storage only.')